In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# STEP 1: LOAD SILVER TABLES

air = spark.table("smart_city.pollution.silver_air_quality")
weather = spark.table("smart_city.pollution.silver_weather")
nasa = spark.table("smart_city.pollution.silver_nasa_power")

print("All Silver tables loaded successfully!")

# STEP 2: AGGREGATE DAILY AIR QUALITY

air_daily = (
    air.groupBy("city", "date")
    .agg(
        F.avg("pm2_5").alias("pm2_5_avg"),
        F.max("pm2_5").alias("pm2_5_max"),
        F.avg("pm10").alias("pm10_avg"),
        F.avg("nitrogen_dioxide").alias("no2_avg"),
        F.avg("sulphur_dioxide").alias("so2_avg"),
        F.avg("carbon_monoxide").alias("co_avg"),
        F.avg("ozone").alias("ozone_avg"),
        F.count("*").alias("air_quality_readings")
    )
)

print("Daily Air Quality aggregation completed!")

All Silver tables loaded successfully!
Daily Air Quality aggregation completed!


In [0]:
# STEP 3: AGGREGATE DAILY WEATHER

weather_daily = (
    weather.groupBy("city", "date")
    .agg(
        F.avg("temperature").alias("weather_temperature_avg"),
        F.avg("humidity").alias("weather_humidity_avg"),
        F.avg("wind_speed").alias("weather_wind_speed_avg"),
        F.sum("precipitation").alias("weather_precipitation_total"),
        F.count("*").alias("weather_readings")
    )
)

print("Daily Weather aggregation completed!")



Daily Weather aggregation completed!


In [0]:
# STEP 4: PREPARE NASA POWER DATA

nasa_daily = (
    nasa.groupBy("city", "date")
    .agg(
        F.avg("temperature_avg").alias("nasa_temperature_avg"),
        F.max("temperature_max").alias("nasa_temperature_max"),
        F.min("temperature_min").alias("nasa_temperature_min"),
        F.avg("humidity").alias("nasa_humidity"),
        F.avg("wind_speed").alias("nasa_wind_speed"),
        F.sum("precipitation").alias("nasa_precipitation"),
        F.count("*").alias("nasa_readings")
    )
)

print("NASA POWER data prepared successfully!")


NASA POWER data prepared successfully!


In [0]:
# STEP 5: INTEGRATE ALL THREE DATASETS

gold_daily = (
    air_daily
    .join(
        weather_daily,
        on=["city", "date"],
        how="full_outer"
    )
    .join(
        nasa_daily,
        on=["city", "date"],
        how="full_outer"
    )
)

gold_daily = gold_daily.withColumn(
    "data_updated_at",
    F.current_timestamp()
)

gold_daily = gold_daily.orderBy("date", "city")

print("All three datasets integrated successfully!")



All three datasets integrated successfully!


In [0]:
# STEP 6: VALIDATE INTEGRATED DATA

print("========== GOLD DATA VALIDATION ==========")

print("Total records:", gold_daily.count())

print("Unique cities:",
      gold_daily.select("city").distinct().count())

print("Unique dates:",
      gold_daily.select("date").distinct().count())

print("Duplicate city-date records:",
      gold_daily.groupBy("city", "date")
      .count()
      .filter(F.col("count") > 1)
      .count())

display(gold_daily.limit(20))



========== GOLD DATA VALIDATION ==========
Total records: 42
Unique cities: 6
Unique dates: 7
Duplicate city-date records: 0


city,date,pm2_5_avg,pm2_5_max,pm10_avg,no2_avg,so2_avg,co_avg,ozone_avg,air_quality_readings,weather_temperature_avg,weather_humidity_avg,weather_wind_speed_avg,weather_precipitation_total,weather_readings,nasa_temperature_avg,nasa_temperature_max,nasa_temperature_min,nasa_humidity,nasa_wind_speed,nasa_precipitation,nasa_readings,data_updated_at
Bengaluru,2026-11-02,19.820833333333336,36.9,21.679166666666664,30.437500000000004,12.262500000000003,809.6666666666666,36.416666666666664,24,22.69583333333333,83.04166666666667,6.199999999999999,10.200000000000001,24,21.42,23.8,18.77,90.0,1.32,4.11,1,2026-11-11T14:19:44.709Z
Chennai,2026-11-02,7.949999999999999,13.8,8.883333333333335,16.249999999999996,11.204166666666666,447.4166666666667,44.625,24,25.254166666666666,90.25,12.516666666666666,27.0,24,26.31,28.62,24.89,89.33,3.17,11.77,1,2026-11-11T14:19:44.709Z
Delhi,2026-11-02,104.91666666666669,136.0,174.975,43.75416666666666,29.841666666666672,1174.0416666666667,85.875,24,23.525000000000002,66.29166666666667,4.466666666666666,0.0,24,23.64,29.98,17.47,46.57,1.46,0.0,1,2026-11-11T14:19:44.709Z
Hyderabad,2026-11-02,17.533333333333335,34.1,18.233333333333334,7.383333333333332,6.766666666666668,398.5833333333333,71.45833333333333,24,24.950000000000003,75.25,9.487499999999999,0.2,24,23.84,27.8,20.59,86.17,3.36,5.0,1,2026-11-11T14:19:44.709Z
Kolkata,2026-11-02,67.79166666666667,93.6,68.45,16.166666666666664,27.399999999999995,681.6666666666666,103.16666666666667,24,27.845833333333335,82.70833333333333,4.016666666666667,2.9,24,26.13,30.37,22.71,87.38,1.79,0.69,1,2026-11-11T14:19:44.709Z
Mumbai,2026-11-02,46.07916666666668,65.2,47.59583333333333,22.75,24.86666666666667,546.25,134.20833333333334,24,29.920833333333334,59.166666666666664,6.404166666666668,0.30000000000000004,24,26.46,31.35,22.6,80.81,1.83,2.95,1,2026-11-11T14:19:44.709Z
Bengaluru,2026-11-03,24.479166666666668,35.6,25.9375,31.916666666666668,12.987499999999999,656.4166666666666,51.291666666666664,24,23.616666666666664,73.91666666666667,3.7708333333333335,1.6,24,23.14,27.65,18.81,85.51,1.25,4.21,1,2026-11-11T14:19:44.709Z
Chennai,2026-11-03,17.425,21.5,20.54583333333333,11.958333333333334,10.287500000000001,428.5416666666667,80.75,24,27.43333333333334,82.08333333333333,5.154166666666666,2.3000000000000003,24,27.05,29.86,24.86,84.45,2.43,5.44,1,2026-11-11T14:19:44.709Z
Delhi,2026-11-03,125.82916666666667,156.0,227.82083333333333,31.404166666666665,25.370833333333326,1028.7916666666667,88.75,24,22.891666666666666,67.5,6.045833333333333,0.0,24,23.66,30.89,16.72,44.4,2.36,0.0,1,2026-11-11T14:19:44.709Z
Hyderabad,2026-11-03,7.683333333333333,11.8,8.241666666666665,7.533333333333334,5.550000000000001,297.125,60.333333333333336,24,24.633333333333336,71.04166666666667,8.733333333333333,0.9,24,23.65,27.8,21.37,88.15,5.05,10.07,1,2026-11-11T14:19:44.709Z


In [0]:
# STEP 7: SAVE GOLD CITY DAILY TABLE

gold_daily.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "smart_city.pollution.gold_city_daily"
    )

print("gold_city_daily created successfully!")

# STEP 8: CREATE POLLUTION HOTSPOTS

hotspots = (
    gold_daily
    .groupBy("city")
    .agg(
        F.avg("pm2_5_avg").alias("average_pm2_5"),
        F.max("pm2_5_max").alias("maximum_pm2_5"),
        F.avg("pm10_avg").alias("average_pm10"),
        F.count("pm2_5_avg").alias("days_with_air_quality_data"),
        F.avg("weather_temperature_avg").alias("average_temperature"),
        F.avg("weather_humidity_avg").alias("average_humidity")
    )
    .filter(F.col("average_pm2_5").isNotNull())
)

ranking = Window.orderBy(
    F.desc("average_pm2_5")
)

hotspots = (
    hotspots
    .withColumn(
        "pollution_rank",
        F.dense_rank().over(ranking)
    )
    .orderBy("pollution_rank")
)

display(hotspots)

hotspots.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "smart_city.pollution.gold_pollution_hotspots"
    )

print("gold_pollution_hotspots created successfully!")


gold_city_daily created successfully!


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


city,average_pm2_5,maximum_pm2_5,average_pm10,days_with_air_quality_data,average_temperature,average_humidity,pollution_rank
Delhi,117.25297619047619,186.7,195.63511904761904,7,23.23095238095238,67.35119047619047,1
Kolkata,97.95119047619048,171.8,105.9589285714286,7,26.59464285714285,75.64285714285714,2
Mumbai,53.044642857142854,107.8,57.6154761904762,7,30.62142857142857,48.607142857142854,3
Hyderabad,20.204761904761906,56.1,22.323214285714283,7,25.16964285714286,66.17857142857143,4
Chennai,16.688095238095237,64.0,20.59047619047619,7,27.46309523809524,81.26785714285714,5
Bengaluru,16.470833333333335,36.9,17.41845238095238,7,23.092857142857138,79.98214285714286,6


gold_pollution_hotspots created successfully!


In [0]:
# STEP 9: CREATE POLLUTANT TRENDS

pollutant_trends = gold_daily.selectExpr(
    "city",
    "date",
    """
    stack(
        6,
        'PM2.5', pm2_5_avg,
        'PM10', pm10_avg,
        'NO2', no2_avg,
        'SO2', so2_avg,
        'CO', co_avg,
        'Ozone', ozone_avg
    ) AS (pollutant, average_value)
    """
).filter(
    F.col("average_value").isNotNull()
)

pollutant_trends = pollutant_trends.orderBy(
    "date", "city", "pollutant"
)

display(pollutant_trends.limit(30))

pollutant_trends.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "smart_city.pollution.gold_pollutant_trends"
    )

print("gold_pollutant_trends created successfully!")



city,date,pollutant,average_value
Bengaluru,2026-11-02,CO,809.6666666666666
Bengaluru,2026-11-02,NO2,30.437500000000004
Bengaluru,2026-11-02,Ozone,36.416666666666664
Bengaluru,2026-11-02,PM10,21.679166666666664
Bengaluru,2026-11-02,PM2.5,19.820833333333336
Bengaluru,2026-11-02,SO2,12.262500000000003
Chennai,2026-11-02,CO,447.4166666666667
Chennai,2026-11-02,NO2,16.249999999999996
Chennai,2026-11-02,Ozone,44.625
Chennai,2026-11-02,PM10,8.883333333333335


gold_pollutant_trends created successfully!


In [0]:
# STEP 10: FINAL GOLD LAYER VERIFICATION

print("========================================")
print("GOLD LAYER COMPLETED SUCCESSFULLY!")
print("========================================")

for table_name in [
    "gold_city_daily",
    "gold_pollution_hotspots",
    "gold_pollutant_trends"
]:

    df = spark.table(
        "smart_city.pollution." + table_name
    )

    print("\nTable:", table_name)
    print("Records:", df.count())
    df.printSchema()
    display(df.limit(5))

print("\nALL GOLD TABLES CREATED SUCCESSFULLY!")


GOLD LAYER COMPLETED SUCCESSFULLY!

Table: gold_city_daily
Records: 42
root
 |-- city: string (nullable = true)
 |-- date: date (nullable = true)
 |-- pm2_5_avg: double (nullable = true)
 |-- pm2_5_max: double (nullable = true)
 |-- pm10_avg: double (nullable = true)
 |-- no2_avg: double (nullable = true)
 |-- so2_avg: double (nullable = true)
 |-- co_avg: double (nullable = true)
 |-- ozone_avg: double (nullable = true)
 |-- air_quality_readings: long (nullable = true)
 |-- weather_temperature_avg: double (nullable = true)
 |-- weather_humidity_avg: double (nullable = true)
 |-- weather_wind_speed_avg: double (nullable = true)
 |-- weather_precipitation_total: double (nullable = true)
 |-- weather_readings: long (nullable = true)
 |-- nasa_temperature_avg: double (nullable = true)
 |-- nasa_temperature_max: double (nullable = true)
 |-- nasa_temperature_min: double (nullable = true)
 |-- nasa_humidity: double (nullable = true)
 |-- nasa_wind_speed: double (nullable = true)
 |-- nasa_p

city,date,pm2_5_avg,pm2_5_max,pm10_avg,no2_avg,so2_avg,co_avg,ozone_avg,air_quality_readings,weather_temperature_avg,weather_humidity_avg,weather_wind_speed_avg,weather_precipitation_total,weather_readings,nasa_temperature_avg,nasa_temperature_max,nasa_temperature_min,nasa_humidity,nasa_wind_speed,nasa_precipitation,nasa_readings,data_updated_at
Bengaluru,2026-11-02,19.820833333333336,36.9,21.679166666666664,30.437500000000004,12.262500000000003,809.6666666666666,36.416666666666664,24,22.69583333333333,83.04166666666667,6.199999999999999,10.200000000000001,24,21.42,23.8,18.77,90.0,1.32,4.11,1,2026-11-11T14:19:47.090Z
Chennai,2026-11-02,7.949999999999999,13.8,8.883333333333335,16.249999999999996,11.204166666666666,447.4166666666667,44.625,24,25.254166666666666,90.25,12.516666666666666,27.0,24,26.31,28.62,24.89,89.33,3.17,11.77,1,2026-11-11T14:19:47.090Z
Delhi,2026-11-02,104.91666666666669,136.0,174.975,43.75416666666666,29.841666666666672,1174.0416666666667,85.875,24,23.525000000000002,66.29166666666667,4.466666666666666,0.0,24,23.64,29.98,17.47,46.57,1.46,0.0,1,2026-11-11T14:19:47.090Z
Hyderabad,2026-11-02,17.533333333333335,34.1,18.233333333333334,7.383333333333332,6.766666666666668,398.5833333333333,71.45833333333333,24,24.950000000000003,75.25,9.487499999999999,0.2,24,23.84,27.8,20.59,86.17,3.36,5.0,1,2026-11-11T14:19:47.090Z
Kolkata,2026-11-02,67.79166666666667,93.6,68.45,16.166666666666664,27.399999999999995,681.6666666666666,103.16666666666667,24,27.845833333333335,82.70833333333333,4.016666666666667,2.9,24,26.13,30.37,22.71,87.38,1.79,0.69,1,2026-11-11T14:19:47.090Z



Table: gold_pollution_hotspots
Records: 6
root
 |-- city: string (nullable = true)
 |-- average_pm2_5: double (nullable = true)
 |-- maximum_pm2_5: double (nullable = true)
 |-- average_pm10: double (nullable = true)
 |-- days_with_air_quality_data: long (nullable = true)
 |-- average_temperature: double (nullable = true)
 |-- average_humidity: double (nullable = true)
 |-- pollution_rank: integer (nullable = true)



city,average_pm2_5,maximum_pm2_5,average_pm10,days_with_air_quality_data,average_temperature,average_humidity,pollution_rank
Delhi,117.25297619047619,186.7,195.63511904761904,7,23.23095238095238,67.35119047619047,1
Kolkata,97.95119047619048,171.8,105.9589285714286,7,26.59464285714285,75.64285714285714,2
Mumbai,53.044642857142854,107.8,57.6154761904762,7,30.62142857142857,48.607142857142854,3
Hyderabad,20.204761904761906,56.1,22.323214285714283,7,25.16964285714286,66.17857142857143,4
Chennai,16.688095238095237,64.0,20.59047619047619,7,27.46309523809524,81.26785714285714,5



Table: gold_pollutant_trends
Records: 252
root
 |-- city: string (nullable = true)
 |-- date: date (nullable = true)
 |-- pollutant: string (nullable = true)
 |-- average_value: double (nullable = true)



city,date,pollutant,average_value
Bengaluru,2026-11-02,CO,809.6666666666666
Bengaluru,2026-11-02,NO2,30.437500000000004
Bengaluru,2026-11-02,Ozone,36.416666666666664
Bengaluru,2026-11-02,PM10,21.679166666666664
Bengaluru,2026-11-02,PM2.5,19.820833333333336



ALL GOLD TABLES CREATED SUCCESSFULLY!
